In [ ]:
pip install ultralytics opencv-python numpy

In [ ]:
import os
import cv2
from ultralytics import YOLO

# -------------------------
# CONFIG
# -------------------------
LOCAL_VIDEO = "waiting_area.mp4"   # path to your input video
OUTPUT_VIDEO = "synopsis.mp4"

CONF_THRESHOLD = 0.4
OUTPUT_FPS = 60  # higher = faster playback

assert os.path.exists(LOCAL_VIDEO), f"Put your video at {LOCAL_VIDEO}"


In [ ]:
# -# Add at the top if using Colab
from google.colab.patches import cv2_imshow
# 2️⃣ Load YOLOv8 Model
# -------------------------
print("Loading YOLO model...")
from ultralytics import YOLO
import cv2

model = YOLO("yolov8n.pt")  # lightweight for POC

# -------------------------
# 3️⃣ Open Video
# -------------------------
cap = cv2.VideoCapture(LOCAL_VIDEO)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
input_fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

OUTPUT_FPS = 60  # faster playback
out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*'mp4v'),
    OUTPUT_FPS,
    (width, height)
)

print("Running inference and creating synopsis...")

frame_count = 0
kept_frames = 0
SKIP = 3  # process every 3rd frame to reduce redundancy

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Skip frames for faster processing/compression
    if frame_count % SKIP != 0:
        continue

    results = model.track(
        frame,
        persist=True,
        conf=CONF_THRESHOLD,
        verbose=False
    )

    # Keep only frames with people
    if len(results[0].boxes) > 0:
        classes = results[0].boxes.cls
        if 0 in classes:  # class 0 = person
            annotated_frame = results[0].plot()
            out.write(annotated_frame)
            kept_frames += 1

           # --- Frame Visualization (first 10 kept frames) ---
            if kept_frames <= 10:
                cv2_imshow(annotated_frame)
                import time
                time.sleep(0.2)

    if frame_count % 200 == 0:
        print(f"Processed {frame_count}/{total_frames} frames...")

cap.release()
out.release()
cv2.destroyAllWindows()



Loading YOLO model...
Running inference and creating synopsis...


Processed 600/78355 frames...
Processed 1200/78355 frames...
Processed 1800/78355 frames...
Processed 2400/78355 frames...
Processed 3000/78355 frames...
Processed 3600/78355 frames...
Processed 4200/78355 frames...
Processed 4800/78355 frames...
Processed 5400/78355 frames...
Processed 6000/78355 frames...
Processed 6600/78355 frames...
Processed 7200/78355 frames...
Processed 7800/78355 frames...
Processed 8400/78355 frames...
Processed 9000/78355 frames...
Processed 9600/78355 frames...
Processed 10200/78355 frames...
Processed 10800/78355 frames...
Processed 11400/78355 frames...
Processed 12000/78355 frames...
Processed 12600/78355 frames...
Processed 13200/78355 frames...
Processed 13800/78355 frames...
Processed 14400/78355 frames...
Processed 15000/78355 frames...
Processed 15600/78355 frames...
Processed 16200/78355 frames...
Processed 16800/78355 frames...
Processed 17400/78355 frames...
Processed 18000/78355 frames...
Processed 18600/78355 frames...
Processed 19200/78355 fra

In [ ]:
# -------------------------
# 4️⃣ Compression Stats
# -------------------------
original_duration = total_frames / input_fps
synopsis_duration = kept_frames / OUTPUT_FPS
compression_ratio = original_duration / synopsis_duration if synopsis_duration > 0 else 0
idle_removed = 100 * (1 - kept_frames / total_frames)

print("\n✅ Done!")
print(f"Original duration: {original_duration/60:.2f} minutes")
print(f"Synopsis duration: {synopsis_duration/60:.2f} minutes")
print(f"Compression ratio: {compression_ratio:.2f}x")
print(f"Idle frames removed: {idle_removed:.2f}%")
print(f"Saved as: {OUTPUT_VIDEO}")


✅ Done!
Original duration: 52.24 minutes
Synopsis duration: 7.25 minutes
Compression ratio: 7.20x
Idle frames removed: 66.67%
Saved as: synopsis.mp4
